# 📊 Customer Shopping Behaviour Analysis
## Data Analytics and Visualization (DAV) Academic Practical

This notebook implements a complete **Data Analytics and Visualization (DAV)** laboratory workflow in Python.
It covers the full DAV syllabus curriculum:

1. **Data Ingestion:** Multi-format loading (CSV, JSON, XML, TXT) and schema validation
2. **Data Preprocessing:** Imputation, deduplication, IQR outlier detection, type casting, and normalization
3. **Data Integration & Relational SQL:** Normalized SQLite database, primary/foreign keys, joins, CTEs, and window functions
4. **Descriptive Statistics:** Measures of central tendency, dispersion, IQR, grouped aggregation, and cross-tabulation
5. **Data Visualization:** Matplotlib, Seaborn, and Plotly graphics
6. **Statistical Hypothesis Testing:** Two-sample t-test, Chi-square independence test, and One-way ANOVA
7. **Regression Analysis:** Purchase amount prediction, baseline evaluation, residual analysis, and academic limitations
8. **Time-Series Analysis:** Authentic retail dated observations, rolling statistics, seasonality, and chronological forecasting
9. **Data Export:** Exporting processed datasets to CSV and JSON formats


In [ ]:
# Setup workspace and imports
import sys
from pathlib import Path

# Add src directory to Python path
REPO_DIR = Path("..").resolve()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from src.data_loader import DataLoader
from src.preprocessor import Preprocessor
from src.database import DatabaseManager
from src.statistics import DescriptiveStatistics, CorrelationAnalyzer, HypothesisTesting
from src.visualization import Visualizer
from src.regression import RegressionPipeline
from src.timeseries import TimeSeriesAnalyzer

print("All DAV analytical modules loaded successfully!")

## 1. Multi-Format Data Collection & Loading
We demonstrate loading data from CSV, JSON, XML, and TXT formats with schema validation and exception handling.

In [ ]:
raw_dir = REPO_DIR / "data" / "raw"

# 1. Load CSV with schema validation
df_csv = DataLoader.load_csv(raw_dir / "customer_shopping_behavior.csv", validate_schema=True)
print(f"CSV Loaded: {df_csv.shape}")

# 2. Load JSON
df_json = DataLoader.load_json(raw_dir / "sample_data.json")
print(f"JSON Loaded: {df_json.shape}")

# 3. Load XML
df_xml = DataLoader.load_xml(raw_dir / "sample_data.xml")
print(f"XML Loaded: {df_xml.shape}")

# 4. Load Delimited TXT
df_txt = DataLoader.load_txt(raw_dir / "sample_data.txt")
print(f"TXT Loaded: {df_txt.shape}")

df_csv.head(3)

## 2. Data Preprocessing & Cleaning
We handle category-level missing ratings imputation, duplicate checks, data type casting, feature engineering, and IQR outlier boundaries.

In [ ]:
proc_dir = REPO_DIR / "data" / "processed"
cleaned_df, cleaning_stats = Preprocessor.run_full_pipeline(df_csv, export_dir=proc_dir)

print("Preprocessing Pipeline Summary:")
for k, v in cleaning_stats.items():
    print(f" - {k}: {v}")

print(f"\nCleaned Dataset Shape: {cleaned_df.shape}")
cleaned_df.info()

## 3. Relational Database Management with SQLite
We normalize the flat customer shopping data into `customers` and `orders` entities, apply primary and foreign key constraints, and execute analytical SQL queries demonstrating joins, aggregations, CTEs, and window functions.

In [ ]:
db_path = REPO_DIR / "data" / "customer_shopping.db"
schema_path = REPO_DIR / "sql" / "schema.sql"
queries_path = REPO_DIR / "sql" / "queries.sql"

db = DatabaseManager(db_path)
db.initialize_schema(schema_path)
n_cust, n_ord = db.populate_from_dataframe(cleaned_df)
print(f"SQLite Database Populated: {n_cust} customers, {n_ord} orders")

# Execute and display sample SQL query: Top 3 products per category (Window Function)
q8_sql = """
WITH item_counts AS (
    SELECT category, item_purchased, COUNT(*) AS total_orders,
           ROW_NUMBER() OVER (PARTITION BY category ORDER BY COUNT(*) DESC) AS item_rank
    FROM orders
    GROUP BY category, item_purchased
)
SELECT item_rank, category, item_purchased, total_orders
FROM item_counts
WHERE item_rank <= 3;
"""
df_q8 = db.execute_query(q8_sql)
df_q8

## 4. Descriptive Statistics & Grouped Aggregations
We compute univariate statistics (mean, median, mode, variance, standard deviation, IQR, skewness) and grouped business metrics.

In [ ]:
# Univariate summary of numeric features
stats_df = DescriptiveStatistics.compute_all_numeric_summaries(cleaned_df)
print("Univariate Summary:")
display(stats_df.T[["count", "mean", "median", "std_dev", "min", "q1", "q3", "max", "iqr", "skewness"]])

# Category grouped revenue summary
cat_summary = DescriptiveStatistics.grouped_spending_summary(cleaned_df, "category")
print("\nCategory Spending Summary:")
display(cat_summary)

## 5. Exploratory Data Visualization
We visualize distributions, boxplots, demographic spending, and correlation heatmaps using Matplotlib and Seaborn.

In [ ]:
# Purchase amount distribution
fig_dist = Visualizer.plot_purchase_distribution(cleaned_df)
plt.show()

# Category boxplots
fig_box = Visualizer.plot_category_boxplots(cleaned_df)
plt.show()

# Demographic spending
fig_demo = Visualizer.plot_demographic_spending(cleaned_df)
plt.show()

# Correlation heatmap
fig_corr = Visualizer.plot_correlation_heatmap(cleaned_df)
plt.show()

## 6. Statistical Hypothesis Testing
We formulate hypotheses ($H_0$ vs $H_1$) and test them using two-sample t-test, Chi-square independence test, and One-way ANOVA.

In [ ]:
# 1. Two-sample t-test: Subscriber vs Non-subscriber purchase amount
ttest_res = HypothesisTesting.test_subscriber_spending_difference(cleaned_df)
print("Test 1:", ttest_res["test_name"])
print("H0:", ttest_res["null_hypothesis"])
print(f"t-statistic = {ttest_res['t_statistic']:.4f}, p-value = {ttest_res['p_value']:.4f}")
print("Result:", ttest_res["interpretation"])
print("-" * 70)

# 2. Chi-square test: Discount applied vs Category independence
chi2_res = HypothesisTesting.test_discount_category_independence(cleaned_df)
print("Test 2:", chi2_res["test_name"])
print("H0:", chi2_res["null_hypothesis"])
print(f"chi2 = {chi2_res['chi2_statistic']:.4f}, p-value = {chi2_res['p_value']:.4f}")
print("Result:", chi2_res["interpretation"])
print("-" * 70)

# 3. One-Way ANOVA: Seasonal spending variation
anova_res = HypothesisTesting.test_seasonal_spending_anova(cleaned_df)
print("Test 3:", anova_res["test_name"])
print("H0:", anova_res["null_hypothesis"])
print(f"F-statistic = {anova_res['f_statistic']:.4f}, p-value = {anova_res['p_value']:.4f}")
print("Result:", anova_res["interpretation"])

## 7. Regression Analysis
We train a Linear Regression model to predict `purchase_amount` while strictly preventing target leakage, establish a baseline, evaluate using MAE, RMSE, and $R^2$, and visualize residuals.

In [ ]:
reg_pipe = RegressionPipeline(cleaned_df)
reg_res = reg_pipe.train(model_type="linear")

print("Model Evaluation Performance:")
print(f"Baseline (Mean)   -> MAE: {reg_res['baseline']['mae']:.2f}, RMSE: {reg_res['baseline']['rmse']:.2f}")
print(f"Linear Regression -> MAE: {reg_res['test_metrics']['mae']:.2f}, RMSE: {reg_res['test_metrics']['rmse']:.2f}, R2: {reg_res['test_metrics']['r2']:.4f}")

print("\nTop 5 Feature Coefficients:")
display(reg_res["coefficients"].head(5))

# Plot Residuals
fig_res = Visualizer.plot_regression_residuals(
    reg_res["predictions"]["y_test"],
    reg_res["predictions"]["y_pred"]
)
plt.show()

print("\nAcademic Finding / Limitation:")
print(reg_res["limitations"])

## 8. Time-Series Analysis on Dated Retail Transactions
Because the synthetic shopping dataset does not possess transaction dates, we demonstrate time-series analytics on authentic date-indexed retail sales (2023-01 to 2024-01).

In [ ]:
ts_path = REPO_DIR / "data" / "timeseries" / "retail_sales_timeseries.csv"
ts_analyzer = TimeSeriesAnalyzer(ts_path)
daily_df = ts_analyzer.compute_rolling_statistics(window_short=7, window_long=30)

# Plot rolling trends
plt.figure(figsize=(11, 5))
plt.plot(daily_df.index, daily_df["daily_total"], alpha=0.3, label="Daily Sales", color="gray")
plt.plot(daily_df.index, daily_df["rolling_mean_7d"], label="7-Day Moving Avg", color="#1f77b4", linewidth=2)
plt.plot(daily_df.index, daily_df["rolling_mean_30d"], label="30-Day Moving Avg", color="#ff7f0e", linewidth=2.5)
plt.title("Retail Sales Time-Series: Daily Sales & Moving Averages", fontsize=13, weight="bold")
plt.xlabel("Date")
plt.ylabel("Sales ($)")
plt.legend()
plt.tight_layout()
plt.show()

# Monthly Seasonality
season_df = ts_analyzer.compute_monthly_seasonality()
print("Monthly Seasonality Index:")
display(season_df)

# Time-ordered forecasting
fc_res = ts_analyzer.run_temporal_forecasting(test_days=60)
print("\nForecasting Performance on 60 Test Days (Chronological):")
print(f"Naive 30d Mean -> MAE: {fc_res['naive_30d_mean']['mae']:.2f}, RMSE: {fc_res['naive_30d_mean']['rmse']:.2f}")
print(f"Linear Trend   -> MAE: {fc_res['linear_trend']['mae']:.2f}, RMSE: {fc_res['linear_trend']['rmse']:.2f}")

## 9. Data Export
We verify that cleaned datasets and analytical summaries are exported to CSV and JSON formats.

In [ ]:
csv_out = REPO_DIR / "data" / "processed" / "customer_shopping_cleaned.csv"
json_out = REPO_DIR / "data" / "processed" / "customer_shopping_cleaned.json"

print(f"Cleaned CSV Export Exists: {csv_out.exists()} ({csv_out.stat().st_size:,} bytes)")
print(f"Cleaned JSON Export Exists: {json_out.exists()} ({json_out.stat().st_size:,} bytes)")
print("DAV Project Practical Complete!")